In [ ]:
from utils.data import ProteinDataset, ProteinPairDataset
import torch as pt
import numpy as np


data = pt.load(f'./data/pbond0_hbond0.pt', weights_only=False)
datalib = ProteinDataset(data)
lib_map = np.arange(len(datalib), dtype=np.int64)
print('number of proteins:', len(lib_map))
pdb2idx = [(data[2][i], i) for i in range(len(data[2]))] # pdb name -> idx
pdb2idx = dict(pdb2idx)
pair_dataset = ProteinPairDataset(datalib, './data/tmalign.out', pdb2idx)

 106973


In [ ]:
import torch.nn as nn
import torch_geometric.nn as gnn
from torch.nn import TransformerEncoder, TransformerEncoderLayer


class ProteinGCN(nn.Module):
    def __init__(self, embed_dim:int=512, hidden_channels:int=256, out_channels:int=128, num_layers:int=3, num_edge_features:int=10,):
        super().__init__()
        self.emb = nn.Embedding(num_embeddings=21, embedding_dim=embed_dim, padding_idx=0,)
        # node_attr占一维
        self.gcn = gnn.GCN(in_channels=embed_dim+num_edge_features, hidden_channels=hidden_channels, 
                           num_layers=num_layers, out_channels=out_channels,)
        self.shared = nn.Sequential(nn.Linear(4*out_channels, out_channels), nn.SiLU(),)
        self.tm_head = nn.Linear(out_channels, 1,)
        self.seq_head = nn.Linear(out_channels, 1,)

        encoder_layer = TransformerEncoderLayer(d_model=embed_dim, nhead=8, dim_feedforward=embed_dim*4, 
                                                activation='gelu', batch_first=True,)
        self.encoder = TransformerEncoder(encoder_layer, num_layers=1)
        self.mlp = nn.Linear(embed_dim, 21)

    def embed(self, seq_mask):
        seq, mask = seq_mask
        embedding = self.emb(seq) # [batch_size, seq_len, emb_dim]
        embedding = embedding * mask.unsqueeze(-1) # mask: [batch_size, seq_len, 1]
        return embedding

    def encode_protein(self, seq, mask, graph):
        x, edge_idx, edge_attr, batch, node2seq = graph.x, graph.edge_index, graph.edge_attr, graph.batch, graph.node2seq
        emb = self.embed((seq, mask))
        B, L, D = emb.shape
        emb_flat = emb.view(-1, D)
        flat_idx = batch * L + node2seq
        node_emb = emb_flat[flat_idx]
        x = pt.cat([node_emb, x], dim=-1)
        x = self.gcn(x, edge_idx, edge_attr=edge_attr, batch=batch)
        x = gnn.global_mean_pool(x, batch)
        return x        

    def forward(self, data, mode:str='pretraining'):
        if mode == 'pretraining':
            seqs_pad, masks_pad = data
            embs = self.emb(seqs_pad) # [batch_size, seq_len, emb_dim]
            embs = self.encoder(embs, src_key_padding_mask=~masks_pad) # [batch_size, seq_len, emb_dim]
            outputs = self.mlp(embs) # [batch_size, seq_len, 21]
            return outputs
        elif mode == 'finetuning':            
            (seqs, masks, graphs), (inv_i, inv_j) = data
            prot_repr = self.encode_protein(seqs, masks, graphs)
            x_i = prot_repr[inv_i]
            x_j = prot_repr[inv_j]
            feature = pt.cat([x_i, x_j, x_i-x_j, x_i*x_j], dim=-1)
            shared = self.shared(feature)
            tm_score = self.tm_head(shared).squeeze(-1)
            seq_score = self.seq_head(shared).squeeze(-1)
            return tm_score, seq_score
        else:
            raise ValueError(f'Unknown mode: {mode}')

In [ ]:
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from utils.data import pair_collate_fun



batch_size = 512
libloader = DataLoader(pair_dataset, batch_size=batch_size, shuffle=False, collate_fn=pair_collate_fun(datalib), num_workers=6)
pair_map = np.arange(len(pair_dataset), dtype=np.int64)
print('len pair_dataset:', len(pair_map))
_, test_map = train_test_split(lib_map, test_size=1024, random_state=42)
train_set = ProteinPairDataset(pair_dataset, mapping=pair_map)
test_set = ProteinDataset(datalib, mapping=test_map)
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, 
                          collate_fn=pair_collate_fun(datalib), drop_last=True, num_workers=6)
test_loader = DataLoader(test_set, batch_size=batch_size*2, shuffle=False, 
                         collate_fn=pair_collate_fun(datalib), num_workers=6)

In [ ]:
from tqdm import tqdm
from utils.tools import gen_embeddings, build_idx, calculate_score,save_model
from dataclasses import dataclass


@dataclass
class TrainConfig:
    epochs: int = 10
    gpu: int = 6
    model_name: str = 'gcn'
    lr: float = 1e-3


class RegressionTrainer:
    def __init__(self, model, config):
        super().__init__()
        self.model = model
        self.config = config
        self.f = open(f'{config.model_name}.txt', 'w')
        self.max_score = 0.0
        self.criterion = nn.SmoothL1Loss()
        self.optimizer = pt.optim.AdamW(self.model.parameters(), lr=config.lr)
        self.scheduler = pt.optim.lr_scheduler.StepLR(self.optimizer, step_size=1, gamma=0.1)

    def train(self, train_loader:DataLoader, test_loader:DataLoader, lib_loader:DataLoader):
        for epoch in range(self.config.epochs):
            print(f'=======================Train_epoch{epoch+1}===========================')
            self.f.write('\nEpoch%d  Training\n' % (epoch+1))
            train_loss = []
            self.model.train()
            for i, batch in enumerate(tqdm(train_loader, unit='batch')):
                prot, inv, score = batch
                prot = [x.to(self.config.gpu) for x in prot]
                inv = [x.to(self.config.gpu) for x in inv]
                score = score.to(self.config.gpu)
                output = self.model((prot, inv))
                tm_score, seq_score = output
                tm_loss = self.criterion(tm_score, score[:, 0])
                seq_loss = self.criterion(seq_score, score[:, 1])
                loss = tm_loss + seq_loss
                self.optimizer.zero_grad()
                loss.backward()
                self.optimizer.step()
                train_loss.append(loss.item())
                if (i+1) % 500 == 0:
                    l = pt.tensor(train_loss).mean()
                    print(f'Epoch [{epoch+1}/{self.config.epochs}], Train Loss: {l:.4f}')
                    self.f.write(f'Epoch [{epoch+1}/{self.config.epochs}], Train Loss: {l:.4f}\n')
                    train_loss = []
            self.scheduler.step()
            print(f'=======================Train_epoch{epoch+1}===========================')
            self.f.write('\nEpoch%d  Testing\n' % (epoch+1))
            embs_lib = gen_embeddings(self.model, lib_loader, self.config.gpu)
            embs_test = gen_embeddings(self.model, test_loader, self.config.gpu)
            I, _ = build_idx(embs_lib, embs_test, self.config.gpu)
            score = calculate_score(test_set, datalib, I)
            print(f'Remote homologous score: {score}')
            self.f.write(f'Remote homologous score: {score}')
            if score > self.max_score:
                self.max_score = score
                save_model(self.model, self.config.model_name)
            print(f'======================================================================')
        self.f.close()   


if __name__ == '__main__':
    config = TrainConfig()
    pt.cuda.set_device(config.gpu)
    model = ProteinGCN().cuda(config.gpu)
    trainer = RegressionTrainer(model, config)
    trainer.train(train_loader, test_loader, libloader)
    pt.cuda.empty_cache()